# Evaluación de Sistemas RAG: Más allá de la Precisión Simple

En este notebook exploraremos cómo evaluar rigurosamente un sistema RAG en producción, cubriendo métricas de retrieval, generativa y de sistema completo que rara vez se enseñan en los tutorials básicos.

## ¿Por qué la evaluación de RAG es compleja?

Evaluar un sistema RAG presenta desafíos únicos:
- Tiene múltiples componentes (retrieval + generación) que pueden fallar independientemente
- La calidad de la generación depende tanto de la recuperación como del LLM
- No siempre hay una respuesta única "correcta"
- Las métricas tradicionales de NLP (BLEU, ROUGE) no capturan bien la utilidad en contexto
- En producción, necesitamos métricas que correlacionen con la satisfacción del usuario y el impacto business

## 1. Evaluación del Componente de Retrieval

Antes de evaluar el sistema completo, debemos asegurarnos de que nuestro recuperador está funcionando correctamente.

### Métricas Clásicas de Information Retrieval

Estas métricas asumen que tenemos consultas con juicios de relevancia (ground truth).

In [1]:
import numpy as np
from typing import List, Dict, Set, Tuple

class RetrievalEvaluator:
    @staticmethod
    def precision_at_k(retrieved: List[str], relevant: Set[str], k: int) -> float:
        """Precision@K: proporción de resultados relevantes en el top-k"""
        if k == 0:
            return 0.0
        retrieved_k = retrieved[:k]
        if not retrieved_k:
            return 0.0
        num_relevant = sum(1 for doc in retrieved_k if doc in relevant)
        return num_relevant / len(retrieved_k)
    
    @staticmethod
    def recall_at_k(retrieved: List[str], relevant: Set[str], k: int) -> float:
        """Recall@K: proporción de documentos relevantes recuperados en el top-k"""
        if not relevant:
            return 0.0
        retrieved_k = retrieved[:k]
        num_relevant = sum(1 for doc in retrieved_k if doc in relevant)
        return num_relevant / len(relevant)
    
    @staticmethod
    def mean_reciprocal_rank(retrieved: List[str], relevant: Set[str]) -> float:
        """MRR: promedio del inverso del rango del primer resultado relevante"""
        for i, doc in enumerate(retrieved):
            if doc in relevant:
                return 1.0 / (i + 1)
        return 0.0
    
    @staticmethod
    def average_precision(retrieved: List[str], relevant: Set[str]) -> float:
        """Average Precision: área bajo la curva de precision-recall"""
        if not relevant:
            return 0.0
        
        num_relevant = 0
        sum_precision = 0.0
        
        for i, doc in enumerate(retrieved):
            if doc in relevant:
                num_relevant += 1
                precision_at_i = num_relevant / (i + 1)
                sum_precision += precision_at_i
        
        return sum_precision / len(relevant) if relevant else 0.0
    
    @staticmethod
    def mean_average_precision(queries_results: List[Tuple[List[str], Set[str]]]) -> float:
        """MAP: promedio de Average Precision across múltiples consultas"""
        if not queries_results:
            return 0.0
        ap_scores = [
            RetrievalEvaluator.average_precision(retrieved, relevant)
            for retrieved, relevant in queries_results
        ]
        return np.mean(ap_scores)


### Ejemplo de uso de métricas de retrieval

Vamos a crear un conjunto de datos de ejemplo para demostrar cómo se calcularían estas métricas.

In [2]:
# Simulamos un escenario de evaluación
queries = [
    "¿Cuál es el proceso mediante el cual las plantas convierten luz en energía?",
    "¿Cómo obtienen energía las células a partir de nutrientes?",
    "¿Qué proceso celular produce dos células hijas idénticas?",
]

# Documentos en nuestra base de conocimientos (simplificado)
documents = {
    'doc1': "La fotosíntesis es el proceso mediante el cual las plantas, algas y algunas bacterias convierten la luz solar en energía química.",
    'doc2': "La respiración celular es el proceso por el cual las células obtienen energía a partir de nutrientes como la glucosa.",
    'doc3': "En la mitosis, las células se dividen para producir dos células hijas idénticas.",
    'doc4': "El ADN contiene las instrucciones genéticas para el desarrollo y funcionamiento de los organismos vivos.",
    'doc5': "La teoría de la evolución explica cómo las especies cambian a lo largo del tiempo.",
}

# Juicios de relevancia (en un escenario real, estos vendrían de anotadores humanos)
relevance_judgments = {
    "¿Cuál es el proceso mediante el cual las plantas convierten luz en energía?": {'doc1'},
    "¿Cómo obtienen energía las células a partir de nutrientes?": {'doc2'},
    "¿Qué proceso celular produce dos células hijas idénticas?": {'doc3'},
}

# Simulamos resultados de un recuperador (ordenados por relevancia decreciente)
retriever_results = {
    "¿Cuál es el proceso mediante el cual las plantas convierten luz en energía?": [
        'doc1', 'doc4', 'doc2', 'doc5', 'doc3'
    ],
    "¿Cómo obtienen energía las células a partir de nutrientes?": [
        'doc2', 'doc1', 'doc4', 'doc3', 'doc5'
    ],
    "¿Qué proceso celular produce dos células hijas idénticas?": [
        'doc3', 'doc2', 'doc1', 'doc5', 'doc4'
    ],
}

# Calculamos métricas para cada consulta
print("MÉTRICAS DE RETRIEVAL POR CONSULTA")
print("=" * 60)

for query in queries:
    retrieved = retriever_results[query]
    relevant = relevance_judgments[query]
    
    print(f"\nConsulta: {query}")
    print(f"Documentos relevantes: {relevant}")
    print(f"Resultados recuperados: {retrieved}")
    print(f"Precision@1: {RetrievalEvaluator.precision_at_k(retrieved, relevant, 1):.3f}")
    print(f"Precision@3: {RetrievalEvaluator.precision_at_k(retrieved, relevant, 3):.3f}")
    print(f"Recall@1: {RetrievalEvaluator.recall_at_k(retrieved, relevant, 1):.3f}")
    print(f"Recall@3: {RetrievalEvaluator.recall_at_k(retrieved, relevant, 3):.3f}")
    print(f"MRR: {RetrievalEvaluator.mean_reciprocal_rank(retrieved, relevant):.3f}")
    print(f"AP: {RetrievalEvaluator.average_precision(retrieved, relevant):.3f}")

# Métricas agregadas
queries_results = [(retriever_results[q], relevance_judgments[q]) for q in queries]
print(f"\nMÉTRICAS AGREGADAS")
print("=" * 60)
print(f"MAP: {RetrievalEvaluator.mean_average_precision(queries_results):.3f}")
print(f"MRR promedio: {np.mean([RetrievalEvaluator.mean_reciprocal_rank(r, rel) for r, rel in queries_results]):.3f}")


MÉTRICAS DE RETRIEVAL POR CONSULTA

Consulta: ¿Cuál es el proceso mediante el cual las plantas convierten luz en energía?
Documentos relevantes: {'doc1'}
Resultados recuperados: ['doc1', 'doc4', 'doc2', 'doc5', 'doc3']
Precision@1: 1.000
Precision@3: 0.333
Recall@1: 1.000
Recall@3: 1.000
MRR: 1.000
AP: 1.000

Consulta: ¿Cómo obtienen energía las células a partir de nutrientes?
Documentos relevantes: {'doc2'}
Resultados recuperados: ['doc2', 'doc1', 'doc4', 'doc3', 'doc5']
Precision@1: 1.000
Precision@3: 0.333
Recall@1: 1.000
Recall@3: 1.000
MRR: 1.000
AP: 1.000

Consulta: ¿Qué proceso celular produce dos células hijas idénticas?
Documentos relevantes: {'doc3'}
Resultados recuperados: ['doc3', 'doc2', 'doc1', 'doc5', 'doc4']
Precision@1: 1.000
Precision@3: 0.333
Recall@1: 1.000
Recall@3: 1.000
MRR: 1.000
AP: 1.000

MÉTRICAS AGREGADAS
MAP: 1.000
MRR promedio: 1.000


## 2. Limitaciones de las Métricas Tradicionales en RAG

Aunque útiles, estas métricas tienen limitaciones importantes para sistemas RAG:

### Problemas con las métricas clásicas:
1. **Requieren juicios de relevancia costosos**: Necesitamos datos anotados por humanos
2. **Enfocan solo en retrieval**: No miden la calidad de la generación
3. **Asumen relevancia binaria**: En la práctica, la relevancia es gradual
4. **No capturan utilidad**: Un documento puede ser relevante pero no útil para responder la consulta
5. **Ignoran el contexto de generación**: No consideran cómo el LLM usa el contexto recuperado

## 3. Métricas Basadas en LLMs para Evaluación

Para superar algunas limitaciones, podemos usar LLMs como jueces para evaluar tanto retrieval como generación.

### Evaluación de Retrieval con LLM-as-a-Judge

En lugar de juicios humanos costosos, podemos pedir a un LLM que evalúe la relevancia de los documentos recuperados.

In [ ]:
class LLMRetrievalEvaluator:
    def __init__(self, llm_client=None):
        """
        Inicializa el evaluador con un cliente LLM (OpenAI, Anthropic, etc.)
        En una implementación real, aquí inicializaríamos el cliente.
        Para este ejemplo, simularemos las respuestas.
        """
        self.llm_client = llm_client
    
    def evaluate_relevance(self, query: str, document: str) -> float:
        """
        Usa un LLM para juzgar la relevancia de un documento para una consulta.
        Devuelve una puntuación entre 0 y 1.
        """
        # En una implementación real, haríamos algo como:
        # prompt = f"""Consulta: {query}\n\nDocumento: {document}\n\n"
        #           "En una escala del 0 al 1, ¿qué tan relevante es este documento para responder la consulta?"
        #           "Responde únicamente con un número entre 0 y 1.""""
        #        response = self.llm_client.generate(prompt)
        #        return float(response.strip())
        
        # Para este ejemplo, simulamos basada en coincidencia de palabras clave
        query_words = set(query.lower().split())
        doc_words = set(document.lower().split())
        overlap = len(query_words.intersection(doc_words))
        return min(overlap / max(len(query_words), 1), 1.0)
    
    def precision_at_k_llm(self, retrieved: List[str], query: str, k: int) -> float:
        """Precision@K usando LLM para juzgar relevancia"""
        if k == 0:
            return 0.0
        retrieved_k = retrieved[:k]
        if not retrieved_k:
            return 0.0
        
        relevance_scores = [
            self.evaluate_relevance(query, doc) for doc in retrieved_k
        ]
        # Consideramos relevante si el score > 0.5 (umbral arbitrario para este ejemplo)
        num_relevant = sum(1 for score in relevance_scores if score > 0.5)
        return num_relevant / len(retrieved_k)
    
    def evaluate_chunk_utilities(self, query: str, chunks: List[str]) -> List[float]:
        """
        Evalúa qué tan útil es cada chunk para generar una respuesta a la consulta.
        Esto va más allá de la relevancia simple: evalúa si el chunk contribuye significativamente.
        """
        utilities = []
        for chunk in chunks:
            # En una implementación real, usaríamos el LLM para juzgar utilidad
            # Simulamos basada en longitud y coincidencia de términos
            utility = min(len(chunk) / 200, 1.0) * 0.5  # Factor de longitud
            
            # Bonus por términos específicos de la consulta
            query_terms = set(query.lower().split())
            chunk_terms = set(chunk.lower().split())
            term_overlap = len(query_terms.intersection(chunk_terms))
            utility += min(term_overlap / max(len(query_terms), 1), 1.0) * 0.5
            
            utilities.append(min(utility, 1.0))
        return utilities
